In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib captum

In [ ]:
import torch
import numpy as np
from transformers import TrainingArguments, Trainer
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from transformers import AutoModel, AutoFeatureExtractor, AutoModelForAudioClassification

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Χρήση συσκευής: {device}")

In [ ]:
model_or = AutoModel.from_pretrained("facebook/data2vec-audio-base-960h").to(device).eval()
model_ft = AutoModelForAudioClassification.from_pretrained("VasilisAsim/unispeech_sat-finetuned-IEMOCAP").to(device).eval()
feature_extractor = AutoFeatureExtractor.from_pretrained("VasilisAsim/unispeech_sat-finetuned-IEMOCAP")

###Φόρτωση συνόλου, ετικετών και speaker independent διαχωρισμός

In [ ]:
from datasets import load_dataset, Audio, DatasetDict
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

dataset = load_dataset("AbstractTTS/IEMOCAP", split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

def merge_and_filter(batch):
    emo = batch["major_emotion"]
    if emo == "excited":
        emo = "happy"
    batch["emotion_4class"] = emo
    return batch

dataset = dataset.map(merge_and_filter)
dataset = dataset.filter(lambda b: b["emotion_4class"] in ["angry", "happy", "neutral", "sad"])

def add_speaker(batch):
    parts = batch["file"].split("_")
    session = parts[0][:5]
    speaker_gender = parts[-1][0]
    batch["speaker_id"] = f"{session}_{speaker_gender}"
    return batch

dataset = dataset.map(add_speaker)

labels = sorted(set(dataset["emotion_4class"]))
num_labels = len(labels)
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for i, name in enumerate(labels)}
print("Κλάσεις:", labels)

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["speaker_id"]))

dataset = DatasetDict({
    "train": dataset.select(train_idx),
    "test": dataset.select(test_idx),
})

train_speakers = set(dataset["train"]["speaker_id"])
test_speakers = set(dataset["test"]["speaker_id"])
print("Επικάλυψη ομιλητών:", train_speakers & test_speakers)

In [ ]:
def extract_embeddings(model, feature_extractor, dataset, num_samples=150, encoder_attr=None):
    probe_model = getattr(model, encoder_attr) if encoder_attr else model
    num_layers = model.config.num_hidden_layers + 1
    layer_embeddings = [[] for _ in range(num_layers)]

    for i in range(num_samples):
        sample = dataset[i]
        audio = sample["audio"]["array"]
        inputs = feature_extractor(
            audio,
            sampling_rate=16000,
            max_length=6*16000,
            truncation=True,
            return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            outputs = probe_model(inputs["input_values"], output_hidden_states=True)
        for layer, h in enumerate(outputs.hidden_states):
            pooled = h.mean(dim=1)
            layer_embeddings[layer].append(pooled.squeeze().cpu().numpy())

    return [np.array(layer) for layer in layer_embeddings]

In [ ]:
def extract_all_layers(model, feature_extractor, dataset, num_samples=150, encoder_attr=None, label_field="labels"):
    embeddings = extract_embeddings(model, feature_extractor, dataset, num_samples=num_samples, encoder_attr=encoder_attr)
    labels = [dataset[i][label_field] for i in range(num_samples)]
    return embeddings, labels

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score
)
import pandas as pd

def evaluate_layer(X_train, y_train, X_test, y_test, num_classes):
    probe = LogisticRegression(max_iter=2000, multi_class="multinomial")
    probe.fit(X_train, y_train)

    y_pred = probe.predict(X_test)
    y_proba = probe.predict_proba(X_test)

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    bal_acc = balanced_accuracy_score(y_test, y_pred)  # = UAR
    uar = bal_acc  # ταυτόσημο, το κρατάμε ως ξεχωριστό όνομα για σαφήνεια στη γραφή

    # Macro AUC (One-vs-Rest) - χρειάζεται one-hot ground truth
    y_test_bin = label_binarize(y_test, classes=range(num_classes))
    try:
        auc = roc_auc_score(y_test_bin, y_proba, average="macro", multi_class="ovr")
    except ValueError:
        auc = np.nan

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "UAR": uar,
        "balanced_accuracy": bal_acc,
        "macro_auc": auc
    }


def run_layer_wise_probing(model, feature_extractor, train_ds, test_ds, encoder_attr=None):
    le = LabelEncoder()
    all_labels_train = train_ds["major_emotion"]
    le.fit(all_labels_train)
    num_classes = len(le.classes_)

    train_layers, train_labels_raw = extract_all_layers(model, feature_extractor, train_ds, encoder_attr=encoder_attr, label_field="major_emotion")
    test_layers, test_labels_raw = extract_all_layers(model, feature_extractor, test_ds, encoder_attr=encoder_attr, label_field="major_emotion")

    y_train = le.transform(train_labels_raw)
    y_test = le.transform(test_labels_raw)

    results = []
    for layer_idx in range(len(train_layers)):
        metrics = evaluate_layer(train_layers[layer_idx], y_train, test_layers[layer_idx], y_test, num_classes)
        metrics["layer"] = layer_idx
        results.append(metrics)

    return pd.DataFrame(results)

In [ ]:
train_dataset = dataset["train"]
test_dataset = dataset["test"]

results_or = run_layer_wise_probing(model_or, feature_extractor, train_dataset, test_dataset, encoder_attr=None)
results_or["checkpoint"] = "pretrained"

results_ft = run_layer_wise_probing(model_ft, feature_extractor, train_dataset, test_dataset, encoder_attr="unispeech_sat")
results_ft["checkpoint"] = "finetuned"

results_all = pd.concat([results_or, results_ft], ignore_index=True)

# Plotting όλων των μετρικών

In [ ]:
metrics_to_plot = ["accuracy", "macro_f1", "UAR", "macro_auc"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, metric in zip(axes.flat, metrics_to_plot):
    for ckpt_name in ["pretrained", "finetuned"]:   # αντί για ["original", "M0_frozen", "M1_partial", "M4_best"]
        group = results_all[results_all["checkpoint"] == ckpt_name]
        ax.plot(group["layer"], group[metric], marker="o", label=ckpt_name)
    ax.set_title(metric)
    ax.set_xlabel("Layer")
    ax.grid(alpha=0.3)

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.02))

plt.tight_layout()
plt.show()